# A guided study

This notebook runs the whole pipeline on a tiny synthetic world: it generates the world, fits
the guidance forecast, runs a guided study, and reads the scorecard. Everything is seeded, so it
gives the same numbers every time.

In [ ]:
import tempfile
from pathlib import Path

import duckdb

from hcsim.guidance.config import GuidanceConfig
from hcsim.guidance.pipeline import run as fit_guidance
from hcsim.studies import scorecard
from hcsim.studies.harness import run_study
from hcsim.studies.profile import StudyProfile
from hcsim.synth.config import preset
from hcsim.synth.world import generate_world, write_world

work = Path(tempfile.mkdtemp())
world = work / "tiny.duckdb"
write_world(generate_world(preset("tiny", seed=5)), world)
with duckdb.connect(str(world), read_only=True) as con:
    fitted = fit_guidance(con, GuidanceConfig())
fitted.artifact.save(work / "guidance")
artifact = fitted.artifact
print(f"guidance fitted through {artifact.snapshot_date}, {artifact.horizon} months ahead")

The study forecasts the year after the world's last snapshot, guided by the forecast, with the
invariants checked and the outcomes summarized.

In [ ]:
profile = StudyProfile.from_mapping(
    {
        "name": "guided",
        "seed": 11,
        "trajectories": 5,
        "months": 12,
        "mode": "guided_forecast",
        "scope": "hcf+sbc",
        "evaluators": ["invariants", "summary", "dispersion"],
    }
)
study = run_study(profile, world, work / "guidance")
print(f"{study.verdict.value}: {len(study.metrics)} metrics")

Every invariant holds by construction: every quota is honored, and the headcount, opening, and
accounting identities are exact.

In [ ]:
card = scorecard.scorecard(study)
card[card["evaluator"] == "invariants"][["metric", "value", "threshold", "passed"]]

In [ ]:
assert card.loc[card["evaluator"] == "invariants", "passed"].all()
card[card["evaluator"] != "invariants"][["evaluator", "metric", "value"]]

Each row also carries the provenance of the run behind it, so a number can always be traced to
the exact world, profile, schema, and guidance:

In [ ]:
card.iloc[0][["world_preset", "world_seed", "profile", "spec", "mode", "scope", "seed", "run_date"]]